# Phase 0 / NB1 — Noise Primitives + Conventions

**Goal of Phase 0:** build one trusted function `assignment -> Aer fidelity` that converts any
layer-wise technology schedule into a physics-grounded fidelity number, independent of EFCL.
This notebook builds the two *atomic* noise channels everything downstream is made of, and
locks the conventions so the Aer harness matches EFCL where it can — and where it can't, we
measure the offset instead of hiding it.

**NB1 scope (this notebook only):**
1. `dephasing_channel(T2, t)` — pure dephasing for an idle qubit of duration `t`.
2. `gate_infidelity_channel(F, n)` — gate error reproducing a reported gate fidelity `F`.
3. Two unit-limit checks + the smoke number (0.29) on the real `AerSimulator` backend.

**What carries into NB2+:** the two functions below are imported as-is. No other notebook
redefines a noise channel.

---

### Conventions locked here

**Idle (decoherence) — agrees with EFCL *exactly*.**
EFCL's idle survival is $p_\text{idle}=\exp(-\Delta t/T)$. Pure dephasing
($T_1\!\to\!\infty$) decays the off-diagonal coherence of a qubit by exactly the same factor
$\exp(-t/T_2)$. So the term that actually drives the motivation results (idle penalties,
parking) matches EFCL by construction, with no free parameter.

**Gate fidelity -> channel — a deliberate choice the original "p = 1-F" plan left ambiguous.**
Qiskit's `depolarizing_error(λ, n)` is $E(\rho)=(1-λ)\rho+λI/d$, $d=2^n$. Measured behaviour:
single-qubit `avg_gate_fid = 1 - λ/2`, so the literal reading **λ = 1-F does *not* make any
fidelity metric equal F** (state fidelity would be $(1+F)/2$). Hardware reports `f1q/f2q` as
**average gate fidelities**, so we set

$$\lambda = (1-F)\,\frac{d}{d-1}.$$

Because depolarizing is unitarily covariant, this makes both the *average gate fidelity* and
the *pure-state fidelity* of every gate equal the config `F` **exactly** (verified below for
1Q and 2Q), and gates compose nearly multiplicatively — two `F=0.99` gates give state
fidelity 0.9802 vs $F^2=0.9801$, the $10^{-4}$ excess being depolarizing's residual overlap
on the failure branch (second-order in infidelity). So the exec term tracks EFCL's
$\prod F$ to first order, and **the channel convention is not an EFCL↔Aer divergence source.**
The real divergences are structural and already known — real-SWAP routing vs the $\gamma$
proxy, the active-fast idle remainder, smooth-max vs exact layer timing — and are quantified
in Phase 2, not here.

> **Decision flag — confirm before NB2.** Default = average-gate-fidelity convention above.
> Alternatives if you prefer: process-fidelity match (`λ=4(1-F)/3` for 2Q) or literal
> `λ=1-F`. In the target idle-heavy regime gate infidelities ($10^{-4}$–$10^{-2}$) are
> dwarfed by idle decoherence, so this choice is second-order on absolutes and irrelevant to
> ranking — but it should be a conscious lock, not a default we drifted into.

In [2]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import thermal_relaxation_error, depolarizing_error
from qiskit.quantum_info import (
    DensityMatrix, state_fidelity, average_gate_fidelity, process_fidelity,
)

import qiskit, qiskit_aer
print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)

qiskit 2.4.2 | aer 0.17.2


## 1. Pure dephasing channel

`thermal_relaxation_error(t1=inf, t2=T2, time=t)` — no amplitude damping ($T_1\to\infty$,
consistent with the configs, which have no $T_1$ field), pure $T_2$ dephasing. Returned as a
`QuantumError`; downstream code appends it with `.to_instruction()` at the exact duration
computed per layer.

In [3]:
def dephasing_channel(T2, t):
    """Pure dephasing (T1 -> inf) for an idle qubit of duration `t` (ns) at coherence T2 (ns).

    Off-diagonal coherence decays by exp(-t/T2), matching EFCL's idle survival exactly.
    Returns a 1-qubit QuantumError; append via `.to_instruction()`.
    """
    if t < 0:
        raise ValueError("idle duration must be >= 0")
    return thermal_relaxation_error(t1=np.inf, t2=float(T2), time=float(t))


def _plus_coherence(channel):
    """Apply a 1-qubit channel to |+> and return the coherence factor |rho01(t)|/|rho01(0)|."""
    rho = DensityMatrix.from_label("+").evolve(channel.to_quantumchannel())
    return abs(rho.data[0, 1]) / 0.5

In [4]:
# Coherence law: factor must equal exp(-t/T2) at every duration.
T2_SC = 80000.0  # ns (SC, from cost_config)
print(f"{'t (ns)':>10} {'t/T2':>8} {'coherence':>12} {'exp(-t/T2)':>12} {'|err|':>10}")
for t in [0.0, 20000.0, 100000.0]:
    coh = _plus_coherence(dephasing_channel(T2_SC, t))
    ref = np.exp(-t / T2_SC)
    print(f"{t:>10.0f} {t/T2_SC:>8.4f} {coh:>12.6f} {ref:>12.6f} {abs(coh-ref):>10.2e}")

# Unit-limit #1: T2 -> inf  =>  coherence preserved (factor 1.0).
coh_inf = _plus_coherence(dephasing_channel(1e18, 100000.0))
print(f"\n[unit-limit #1] T2->inf, t=1e5 ns: coherence = {coh_inf:.8f}  (expect 1.0)")

    t (ns)     t/T2    coherence   exp(-t/T2)      |err|
         0   0.0000     1.000000     1.000000   2.22e-16
     20000   0.2500     0.778801     0.778801   1.11e-16
    100000   1.2500     0.286505     0.286505   0.00e+00

[unit-limit #1] T2->inf, t=1e5 ns: coherence = 1.00000000  (expect 1.0)


## 2. Gate infidelity channel

Depolarizing channel whose **average gate fidelity** equals the reported `F`
($\lambda=(1-F)\,d/(d-1)$). Works for 1Q and 2Q (and the move/comm channels in NB4, which
are the same depolarizing form keyed on `f_move`/`f_comm`).

In [5]:
def gate_infidelity_channel(F, num_qubits):
    """Depolarizing channel with average gate fidelity == F (hardware convention).

    lambda = (1-F) * d/(d-1),  d = 2**num_qubits.
    Used for 1Q/2Q gate errors and, in NB4, for inter-QPU comm (f_comm) and move (f_move).
    Returns a QuantumError; append via `.to_instruction()`.
    """
    if not (0.0 <= F <= 1.0):
        raise ValueError("F must be in [0, 1]")
    d = 2 ** num_qubits
    lam = (1.0 - F) * d / (d - 1.0)
    return depolarizing_error(lam, num_qubits)

In [6]:
# Convention check: avg gate fidelity of the channel must equal the config F exactly.
print(f"{'F':>8} {'n':>3} {'avg_gate_fid':>14} {'match':>7}")
for F, n in [(0.9999, 1), (0.9995, 1), (0.9990, 2), (0.9970, 2), (0.9900, 1)]:
    agf = average_gate_fidelity(gate_infidelity_channel(F, n).to_quantumchannel())
    print(f"{F:>8.4f} {n:>3} {agf:>14.8f} {str(np.isclose(agf, F)):>7}")

# Unit-limit #2: F=1 -> identity channel.
agf1 = average_gate_fidelity(gate_infidelity_channel(1.0, 1).to_quantumchannel())
print(f"\n[unit-limit #2] F=1: avg_gate_fid = {agf1:.8f}  (expect 1.0)")

# Per-gate state fidelity also equals F exactly (depolarizing is unitarily covariant),
# and gates compose ~multiplicatively -> exec term tracks EFCL's product of F.
from qiskit.quantum_info import random_statevector
print("\nper-gate pure-state fidelity (should equal F):")
for F, n in [(0.99, 1), (0.997, 2)]:
    sop = gate_infidelity_channel(F, n).to_quantumchannel()
    psi = random_statevector(2 ** n, seed=3)
    sf = state_fidelity(DensityMatrix(psi).evolve(sop), DensityMatrix(psi))
    print(f"  F={F} n={n} -> state_fid={sf:.6f}")
sop = gate_infidelity_channel(0.99, 1).to_quantumchannel()
psi = random_statevector(2, seed=1); rho = DensityMatrix(psi)
sf2 = state_fidelity(rho.evolve(sop).evolve(sop), rho)
print(f"  composition: two F=0.99 1Q gates -> {sf2:.4f}  vs  F^2={0.99**2:.4f}  "
      f"(channel convention is NOT a divergence source)")

       F   n   avg_gate_fid   match
  0.9999   1     0.99990000    True
  0.9995   1     0.99950000    True
  0.9990   2     0.99900000    True
  0.9970   2     0.99700000    True
  0.9900   1     0.99000000    True

[unit-limit #2] F=1: avg_gate_fid = 1.00000000  (expect 1.0)

per-gate pure-state fidelity (should equal F):
  F=0.99 n=1 -> state_fid=0.990000
  F=0.997 n=2 -> state_fid=0.997000
  composition: two F=0.99 1Q gates -> 0.9802  vs  F^2=0.9801  (channel convention is NOT a divergence source)


## 3. Smoke number on the real backend

The 0.29 go/no-go (NB6 check #1) on the actual `AerSimulator(method="density_matrix")`
backend the harness uses — not just `quantum_info` evolution. One SC qubit prepared in |+>
idles through one TI 2Q-gate duration (100 µs) at $T_2^{SC}=80$ µs, so $t/T_2=1.25$ and
coherence should be $\exp(-1.25)\approx0.2865$.

In [7]:
def idle_coherence_on_aer(T2, t_idle):
    """Prepare |+>, append one explicit idle dephasing of duration t_idle, read coherence
    off the AerSimulator density_matrix backend (the path NB4/NB5 use)."""
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.append(dephasing_channel(T2, t_idle).to_instruction(), [0])
    qc.save_density_matrix()
    res = AerSimulator(method="density_matrix").run(qc).result()
    rho = DensityMatrix(res.data(0)["density_matrix"])
    return abs(rho.data[0, 1]) / 0.5

T2_SC, T_TI_2Q = 80000.0, 100000.0
coh = idle_coherence_on_aer(T2_SC, T_TI_2Q)
print(f"SC |+> idle through one TI 2Q gate: coherence = {coh:.6f}  "
      f"(expect exp(-1.25) = {np.exp(-1.25):.6f})")

SC |+> idle through one TI 2Q gate: coherence = 0.286505  (expect exp(-1.25) = 0.286505)


## 4. Checkpoint — NB1 go/no-go

All asserts must pass before building NB2. NB2 imports `dephasing_channel` and
`gate_infidelity_channel` unchanged.

In [8]:
def _checkpoint():
    # 1. coherence law exact at the smoke point
    assert np.isclose(_plus_coherence(dephasing_channel(80000.0, 100000.0)),
                      np.exp(-1.25), atol=1e-6), "dephasing coherence law wrong"
    # 2. unit-limit #1: T2 -> inf preserves coherence
    assert np.isclose(_plus_coherence(dephasing_channel(1e18, 1e5)), 1.0, atol=1e-9), \
        "T2->inf must preserve coherence"
    # 3. avg gate fidelity convention exact for 1Q and 2Q
    for F, n in [(0.9999, 1), (0.9990, 2), (0.9900, 1)]:
        agf = average_gate_fidelity(gate_infidelity_channel(F, n).to_quantumchannel())
        assert np.isclose(agf, F), f"avg_gate_fid != F for F={F}, n={n}"
    # 4. unit-limit #2: F=1 -> identity
    assert np.isclose(
        average_gate_fidelity(gate_infidelity_channel(1.0, 1).to_quantumchannel()),
        1.0), "F=1 must be identity"
    # 5. smoke number on the real Aer backend
    assert np.isclose(idle_coherence_on_aer(80000.0, 100000.0),
                      np.exp(-1.25), atol=1e-6), "Aer-backend smoke number wrong"
    return True

print("NB1 CHECKPOINT PASSED" if _checkpoint() else "FAILED")

NB1 CHECKPOINT PASSED
